## Participants
- Eder Tarifa Fernández
- Zakaria Lasry Sahraoui

Grupo K

## Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
import math

## Data

In [2]:
train = pd.read_csv('data/train_final.csv')
train.head()

,review_id,user_id,business_id,target,useful,funny,cool,useful_user,funny_user,cool_user,...,cat_Event_Planning_&_Ser,cat_American_(Traditiona,cat_Sandwiches,cat_Active_Life,cat_Pizza,cat_Coffee_&_Tea,cat_Fast_Food,cat_American_(New),cat_Breakfast_&_Brunch,cat_Hotels_&_Travel
0,ZZO43qKB-s65zplC8RfJqw,-1BSu2dt_rOAqllw9ZDXtA,smkZq4G1AOm4V6p3id5sww,5.0,0,0,0,7.0,3.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,vojXOF_VOgvuKD95gCO8_Q,xpe178ng_gj5X6HgqtOing,96_c_7twb7hYRZ9HHrq01g,1.0,2,0,1,37.0,1.0,2.0,...,0,0,0,0,0,0,0,0,0,0
2,KwxdbiseRlIRNzpgvyjY0Q,axbaerf2Fk92OB4b9_peVA,e0AYjKfSF0DL-5C1CpOq6Q,4.0,0,0,0,31.0,6.0,2.0,...,0,0,0,0,0,0,0,0,0,0
3,3mwoBcTy-2gMh0L91uaIeA,_GOiybb0rImYKJfwyxEaGg,vF-uptiQ34pVLHJKzPHUlA,5.0,0,0,0,36.0,9.0,9.0,...,0,0,0,0,0,0,0,0,0,0
4,XfWf7XsBWs3kYyYq7Ns1ZQ,ojWKg3B5pH3ncAsxun3kUw,X28XK71RuEXPapeyUOwNzg,5.0,10,4,7,189.0,28.0,133.0,...,0,1,0,0,0,0,0,0,0,0


In [3]:
train.shape

(967784, 45)

In [4]:
train.columns

Index(['review_id', 'user_id', 'business_id', 'target', 'useful', 'funny',
       'cool', 'useful_user', 'funny_user', 'cool_user', 'date', 'year',
       'month', 'day', 'stars_business', 'review_count_business',
       'review_count', 'fans', 'average_stars', 'elite_count', 'friend_count',
       'days_yelping', 'total_compliments', 'is_open', 'postal_code',
       'cat_Restaurants', 'cat_Food', 'cat_Shopping', 'cat_Beauty_&_Spas',
       'cat_Home_Services', 'cat_Nightlife', 'cat_Health_&_Medical',
       'cat_Local_Services', 'cat_Bars', 'cat_Automotive',
       'cat_Event_Planning_&_Ser', 'cat_American_(Traditiona',
       'cat_Sandwiches', 'cat_Active_Life', 'cat_Pizza', 'cat_Coffee_&_Tea',
       'cat_Fast_Food', 'cat_American_(New)', 'cat_Breakfast_&_Brunch',
       'cat_Hotels_&_Travel'],
      dtype='object')

In [5]:
train.rename(columns={'target': 'label'}, inplace=True)

In [6]:
train_data, test_data = train_test_split(train, test_size=0.1, random_state=42, stratify=train["label"])

In [2]:
# check for gpu
import torch
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

True
Using device: cuda


## Functions

In [3]:
def predictor_nn(model, X_test, rounded=True):
    model.eval()
    with torch.no_grad():
        inputs = torch.tensor(X_test.values, dtype=torch.float32).to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
    if rounded:
        predicted = torch.round(predicted.float(), decimals=0)
    return pd.DataFrame({'review_id': X_test.review_id, 'stars': predicted.cpu().numpy()})

def save_predictions(predictions, filename='predictions.csv'):
    folder = 'predictions'
    predictions.to_csv(f'{folder}/{filename}', index=False)

## Models

### AutoGluon baseline

In [4]:
from autogluon.tabular import TabularDataset, TabularPredictor

#### 3 features left

In [8]:
# train with gpu
predictor = TabularPredictor(label="label", eval_metric="mae", problem_type="regression").fit(train_data, presets="medium_quality", ag_args_fit={'num_gpus': 1})

No path specified. Models will be saved in: "AutogluonModels/ag-20260410_075357"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 8.00/8.00 GB
Total GPU Memory:   Free: 8.00 GB, Allocated: 0.00 GB, Total: 8.00 GB
GPU Count:          1
Memory Avail:       8.16 GB / 11.55 GB (70.6%)
Disk Space Avail:   774.53 GB / 1006.85 GB (76.9%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
	Consider setting `time_limit` to ensure training finishes within an expected duration or experiment with a small portion of `train_data` to identify an ideal `presets` and `hyperparameters` configuration.
Beginning AutoGluon training ...
AutoGluon wil

[1000]	valid_set's l1: 0.790289
[2000]	valid_set's l1: 0.788674
[3000]	valid_set's l1: 0.788506
[4000]	valid_set's l1: 0.788203
[5000]	valid_set's l1: 0.788062
[6000]	valid_set's l1: 0.787948
[7000]	valid_set's l1: 0.787712
[8000]	valid_set's l1: 0.787691
[9000]	valid_set's l1: 0.787471
[10000]	valid_set's l1: 0.787474


	-0.7873	 = Validation score   (-mean_absolute_error)
	132.06s	 = Training   runtime
	1.48s	 = Validation runtime
Fitting model: LightGBM ...
	Fitting with cpus=6, gpus=1, mem=0.9/7.6 GB


[1000]	valid_set's l1: 0.783955
[2000]	valid_set's l1: 0.782787


	-0.7825	 = Validation score   (-mean_absolute_error)
	35.54s	 = Training   runtime
	0.16s	 = Validation runtime
Fitting model: RandomForestMSE ...
	To force training the model, specify the model hyperparameter "ag.max_memory_usage_ratio" to a larger value (currently 1.0, set to >=1.11 to avoid the error)
		To set the same value for all models, do the following when calling predictor.fit: `predictor.fit(..., ag_args_fit={"ag.max_memory_usage_ratio": VALUE})`
		Setting "ag.max_memory_usage_ratio" to values above 1 may result in out-of-memory errors. You may consider using a machine with more memory as a safer alternative.
	Not enough memory to train RandomForestMSE... Skipping this model.
Fitting model: CatBoost ...
	Fitting with cpus=6, gpus=1, mem=1.3/7.6 GB
	Training CatBoost with GPU, note that this may negatively impact model quality compared to CPU training.
Default metric period is 5 because MAE is/are not implemented for GPU
	-0.7558	 = Validation score   (-mean_absolute_error)


[1000]	valid_set's l1: 0.784051
[2000]	valid_set's l1: 0.783827


	-0.7836	 = Validation score   (-mean_absolute_error)
	43.3s	 = Training   runtime
	0.21s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ...
	Fitting 1 model on all data | Fitting with cpus=12, gpus=0, mem=0.0/6.8 GB
	Ensemble Weights: {'NeuralNetTorch': 1.0}
	-0.694	 = Validation score   (-mean_absolute_error)
	0.05s	 = Training   runtime
	0.0s	 = Validation runtime
AutoGluon training complete, total runtime = 2418.86s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 200899.5 rows/s (8711 batch size)
TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/home/eder/projects/recommender-systems/competition2/AutogluonModels/ag-20260410_075357")


In [9]:
predictor.fit_summary()

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val          eval_metric  pred_time_val     fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0       NeuralNetTorch  -0.694005  mean_absolute_error       0.042363  1452.958750                0.042363        1452.958750            1       True          6
1  WeightedEnsemble_L2  -0.694005  mean_absolute_error       0.043360  1453.006358                0.000997           0.047609            2       True          8
2             CatBoost  -0.755844  mean_absolute_error       0.003300     6.697860                0.003300           6.697860            1       True          3
3             LightGBM  -0.782547  mean_absolute_error       0.163131    35.535436                0.163131          35.535436            1       True          2
4              XGBoost  -0.783190  mean_absolute_error       0.022417     8.805835                0.022417           8.805835        

/home/eder/miniconda3/envs/master312/lib/python3.12/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"')


{'model_types': {'LightGBMXT': 'LGBModel',
  'LightGBM': 'LGBModel',
  'CatBoost': 'CatBoostModel',
  'NeuralNetFastAI': 'NNFastAiTabularModel',
  'XGBoost': 'XGBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'LightGBMLarge': 'LGBModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'LightGBMXT': -0.7873389002527166,
  'LightGBM': -0.7825467686622483,
  'CatBoost': -0.7558439205782433,
  'NeuralNetFastAI': -0.7865110058322482,
  'XGBoost': -0.7831904738987508,
  'NeuralNetTorch': -0.6940052676058543,
  'LightGBMLarge': -0.7835668659609805,
  'WeightedEnsemble_L2': -0.6940052676058543},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'LightGBMXT': ['LightGBMXT'],
  'LightGBM': ['LightGBM'],
  'CatBoost': ['CatBoost'],
  'NeuralNetFastAI': ['NeuralNetFastAI'],
  'XGBoost': ['XGBoost'],
  'NeuralNetTorch': ['NeuralNetTorch'],
  'LightGBMLarge': ['LightGBMLarge'],
  'WeightedEnsemble_L2': ['WeightedEnsemble_L2']},
 'model_fit_times': {'Li

In [10]:
predictor.evaluate_predictions(y_true=test_data["label"], y_pred=predictor.predict(test_data))

{'mean_absolute_error': -0.6933011567995778,
 'root_mean_squared_error': np.float64(-1.1748247019245306),
 'mean_squared_error': -1.3802130802520622,
 'r2': 0.3694331542375787,
 'pearsonr': 0.6551795811306441,
 'median_absolute_error': -0.0070421695709228516}

#### Good

In [10]:
# Esta versión tiene las columnas de cool, funny y useful de train que no se habian incorporado por error
predictor = TabularPredictor(label="label", eval_metric="mae", problem_type="regression").fit(train_data, presets="medium_quality", ag_args_fit={'num_gpus': 1})

No path specified. Models will be saved in: "AutogluonModels/ag-20260410_095708"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 8.00/8.00 GB
Total GPU Memory:   Free: 8.00 GB, Allocated: 0.00 GB, Total: 8.00 GB
GPU Count:          1
Memory Avail:       7.94 GB / 11.55 GB (68.7%)
Disk Space Avail:   774.25 GB / 1006.85 GB (76.9%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
	Consider setting `time_limit` to ensure training finishes within an expected duration or experiment with a small portion of `train_data` to identify an ideal `presets` and `hyperparameters` configuration.
Beginning AutoGluon training ...
AutoGluon wil

[1000]	valid_set's l1: 0.748102
[2000]	valid_set's l1: 0.743545
[3000]	valid_set's l1: 0.74172
[4000]	valid_set's l1: 0.740662
[5000]	valid_set's l1: 0.739789
[6000]	valid_set's l1: 0.739741
[7000]	valid_set's l1: 0.73939
[8000]	valid_set's l1: 0.738811
[9000]	valid_set's l1: 0.738719
[10000]	valid_set's l1: 0.738313


	-0.7383	 = Validation score   (-mean_absolute_error)
	182.52s	 = Training   runtime
	2.53s	 = Validation runtime
Fitting model: LightGBM ...
	Fitting with cpus=6, gpus=1, mem=0.9/7.0 GB


[1000]	valid_set's l1: 0.746783


	-0.7457	 = Validation score   (-mean_absolute_error)
	72.8s	 = Training   runtime
	0.59s	 = Validation runtime
Fitting model: RandomForestMSE ...
	To force training the model, specify the model hyperparameter "ag.max_memory_usage_ratio" to a larger value (currently 1.0, set to >=1.19 to avoid the error)
		To set the same value for all models, do the following when calling predictor.fit: `predictor.fit(..., ag_args_fit={"ag.max_memory_usage_ratio": VALUE})`
		Setting "ag.max_memory_usage_ratio" to values above 1 may result in out-of-memory errors. You may consider using a machine with more memory as a safer alternative.
	Not enough memory to train RandomForestMSE... Skipping this model.
Fitting model: CatBoost ...
	Fitting with cpus=6, gpus=1, mem=1.2/7.0 GB
	Training CatBoost with GPU, note that this may negatively impact model quality compared to CPU training.
Default metric period is 5 because MAE is/are not implemented for GPU
	-0.7243	 = Validation score   (-mean_absolute_error)
	

In [11]:
predictor.fit_summary()

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val          eval_metric  pred_time_val     fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0       NeuralNetTorch  -0.627015  mean_absolute_error       0.118394  5560.269187                0.118394        5560.269187            1       True          6
1  WeightedEnsemble_L2  -0.627015  mean_absolute_error       0.119473  5560.316182                0.001079           0.046995            2       True          8
2             CatBoost  -0.724344  mean_absolute_error       0.113650    22.064477                0.113650          22.064477            1       True          3
3              XGBoost  -0.731059  mean_absolute_error       0.087644    29.573171                0.087644          29.573171            1       True          5
4      NeuralNetFastAI  -0.737494  mean_absolute_error       0.118776   619.346309                0.118776         619.346309        

/home/eder/miniconda3/envs/master312/lib/python3.12/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"')


{'model_types': {'LightGBMXT': 'LGBModel',
  'LightGBM': 'LGBModel',
  'CatBoost': 'CatBoostModel',
  'NeuralNetFastAI': 'NNFastAiTabularModel',
  'XGBoost': 'XGBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'LightGBMLarge': 'LGBModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'LightGBMXT': -0.738300207138041,
  'LightGBM': -0.7456960294319991,
  'CatBoost': -0.7243443036952504,
  'NeuralNetFastAI': -0.7374935798879853,
  'XGBoost': -0.7310589809886296,
  'NeuralNetTorch': -0.6270152386369859,
  'LightGBMLarge': -0.7375958239194166,
  'WeightedEnsemble_L2': -0.6270152386369859},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'LightGBMXT': ['LightGBMXT'],
  'LightGBM': ['LightGBM'],
  'CatBoost': ['CatBoost'],
  'NeuralNetFastAI': ['NeuralNetFastAI'],
  'XGBoost': ['XGBoost'],
  'NeuralNetTorch': ['NeuralNetTorch'],
  'LightGBMLarge': ['LightGBMLarge'],
  'WeightedEnsemble_L2': ['WeightedEnsemble_L2']},
 'model_fit_times': {'Lig

In [54]:
predictor.evaluate_predictions(y_true=test_data["label"], y_pred=predictor.predict(test_data))

{'mean_absolute_error': -0.6394110887361699,
 'root_mean_squared_error': np.float64(-1.1186028674510708),
 'mean_squared_error': -1.2512723750697579,
 'r2': 0.42834125684905333,
 'pearsonr': 0.6957110105315124,
 'median_absolute_error': -0.0034494400024414062}

In [ ]:
test = pd.read_csv('data/test_final.csv')
preds = predictor.predict(test)
preds = pd.DataFrame({'review_id': test.review_id, 'stars': preds})
save_predictions(preds, 'AutoGluonNeuralNetTorch.csv')

In [ ]:
preds.stars = round(preds.stars, 0)
save_predictions(preds, 'AutoGluonNeuralNetTorchRounded.csv')

In [ ]:
test

In [5]:
test = pd.read_csv('data/test_final.csv')
predictor = TabularPredictor.load('AutogluonModels/ag-20260410_095708')
preds = predictor.predict(test)
preds = pd.DataFrame({'review_id': test.review_id, 'stars': preds})
preds.stars = round(preds.stars, 0)
save_predictions(preds, 'AutoGluonNeuralNetTorchRoundedGood.csv')


In [6]:
res1 = pd.read_csv('predictions/AutoGluonNeuralNetTorchRoundedGood.csv')
res2 = pd.read_csv('predictions/AutoGluonNeuralNetTorchRounded.csv')

In [7]:
# Calculate the mae of column "stars" in res1 and res2
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(res1["stars"], res2["stars"])
print(f"MAE: {mae}")

MAE: 0.22338191506033536


In [8]:
# calculate the accumulated distance of each prediction in res1 and res2
accumulated_distance = np.sum(np.abs(res1["stars"] - res2["stars"]))
print(f"Accumulated distance: {accumulated_distance}")

Accumulated distance: 92651.0


#### One hot encoded data

In [4]:
train = pd.read_csv('data/train_final_ohe.csv')
train.shape

(967784, 45)

In [ ]:
predictor = TabularPredictor(label="label", eval_metric="mae", problem_type="regression").fit(train_data, presets="medium_quality", ag_args_fit={'num_gpus': 1})

### DeepFM

In [ ]:
from models import RecVAE
train_df = pd.read_csv("train_reviews.csv")

recommender = RecVAE(use_gpu=True, verbose=True)
recommender.fit(train_df)

print(recommender.predict(user_id="Ha3iJu77CxlrFm-vQRs_8g",
                          business_id="tnhfDv5Il8EaGSXZGiuQGg"))

print(recommender.recommend(user_id="Ha3iJu77CxlrFm-vQRs_8g", k=5))

## V2
### PREPROCESAMIENTO
Esta versión hereda todo el motor de preprocesamiento de características de la V1 (aplanamiento de atributos y tratamiento híbrido de categorías), pero abandona el split aleatorio para adoptar un marco de evaluación del mundo real, incorporando la identidad de los actores clave.

- Split Temporal Estricto: Partición cronológica basada en date_num (70% Train, 15% Val, 15% Test). Esto garantiza que el modelo aprenda del pasado para predecir el futuro, eliminando el sesgo de "viaje en el tiempo" (Data Leakage) presente en la V1.

- Inclusión de Identidades (user_id y business_id): A diferencia de la V1 (donde se descartaban), en esta versión se agregan explícitamente como variables de entrenamiento. Esto permite que el modelo genere embeddings de identidad, reconociendo y capturando los patrones de comportamiento de usuarios y negocios recurrentes.

- Limpieza de Control Temporal: Se elimina de la memoria únicamente la columna técnica date_num tras realizar la partición cronológica, evitando que la red neuronal use el timestamp numérico como un atajo o regla de decisión simplista.

In [7]:
import preprocess_autogluon2
import pandas as pd
# %run preprocess_autogluon2.py
train = pd.read_parquet('data/train_ag.parquet')
train

,target,user_id,business_id,date_num,review_useful,review_funny,review_cool,date,review_year,review_month,...,attr_RestaurantsCounterService,attr_BusinessParking,attr_Music,attr_DietaryRestrictions_dairy-free,attr_DietaryRestrictions_gluten-free,attr_DietaryRestrictions_vegan,attr_DietaryRestrictions_kosher,attr_DietaryRestrictions_halal,attr_DietaryRestrictions_soy-free,attr_DietaryRestrictions_vegetarian
0,5.0,-1BSu2dt_rOAqllw9ZDXtA,smkZq4G1AOm4V6p3id5sww,1475250572,0,0,0,2016-09-30 15:49:32,2016,9,...,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.0,xpe178ng_gj5X6HgqtOing,96_c_7twb7hYRZ9HHrq01g,1607524791,2,0,1,2020-12-09 14:39:51,2020,12,...,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,axbaerf2Fk92OB4b9_peVA,e0AYjKfSF0DL-5C1CpOq6Q,1378311591,0,0,0,2013-09-04 16:19:51,2013,9,...,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5.0,_GOiybb0rImYKJfwyxEaGg,vF-uptiQ34pVLHJKzPHUlA,1551529454,0,0,0,2019-03-02 12:24:14,2019,3,...,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5.0,ojWKg3B5pH3ncAsxun3kUw,X28XK71RuEXPapeyUOwNzg,1587666389,10,4,7,2020-04-23 18:26:29,2020,4,...,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
967779,5.0,iZAAjkPZ0sopzfajcfdOUg,2PvPsZ3KRFCtHbQkNHvGpg,1587310932,1,0,0,2020-04-19 15:42:12,2020,4,...,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
967780,4.0,AN8XscFSH1jctLSqAQ9-bA,-K0zTgGyxo-AeSkcV0IVaA,1392056823,0,0,0,2014-02-10 18:27:03,2014,2,...,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
967781,5.0,qtgODIPIsKsH2j3rItF9Tw,eZCt3doDaA-l4sg_OM67YQ,1563977992,0,0,1,2019-07-24 14:19:52,2019,7,...,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
967782,5.0,gu7nU1IM7U3lLwUbqLeiDQ,ww3YJXu5c18aGZXWmm00qg,1320734285,0,0,0,2011-11-08 06:38:05,2011,11,...,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### ENTRENAMIENTO

In [ ]:
import train_autogluon2
%run train_autogluon2.py

c:\Users\kzzazzk\OneDrive\Documentos\MAADM\2º Cuatrimestre\2S\RECSYS\recommender-systems\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



█████████████████████████████████████████████████████████████████
  AUTOGLUON — SISTEMA DE RECOMENDACIÓN YELP (v2)
█████████████████████████████████████████████████████████████████

  Cargando datos y haciendo splits temporales...

  Splits por fecha (date_num ascendente):
    Train: [0:677448] = 677,448 reviews (70%)
    Val:   [677448:822616] = 145,168 reviews (15%)
    Test:  [822616:967784] = 145,168 reviews (15%)
  Sanitizando para AutoGluon...


Preset alias specified: 'high_v150' maps to 'high_quality_v150'.
Verbosity: 2 (Standard Logging)



  Estadísticas Target:
    Train — min:1  max:5  mean:3.764  std:1.432
    Val   — min:1  max:5  mean:3.782  std:1.541

  Columnas TEXT (NLP): ['categories']
  Columnas categóricas (inferidas): []

  Iniciando entrenamiento (time_limit=172800s = 48.0h)...


=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.10
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.11.0+cu130
CUDA Version:       13.0
GPU Memory:         GPU 0: 8.00/8.00 GB
Total GPU Memory:   Free: 8.00 GB, Allocated: 0.00 GB, Total: 8.00 GB
GPU Count:          1
Memory Avail:       13.72 GB / 31.90 GB (43.0%)
Disk Space Avail:   191.90 GB / 930.64 GB (20.6%)
Presets specified: ['high_v150']
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=0, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 172800s
AutoGluon will save models to "c:\Users\kzzazzk\OneDrive\Documentos\MAADM\2º Cuatrimestre\2S\RECSYS\recommender-systems\competition2\models2\autogluon_1776010975"
Train Data Rows:    677448
Train Data Columns: 138
Tuning Data Rows:    145168
Tuning Data Columns: 138
Label Column:       target
Problem Type:       regres

### INFERENCIA

In [ ]:
import pandas as pd
import os
from autogluon.tabular import TabularPredictor, TabularDataset

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
MODEL_PATH = "models2/autogluon_1776010975"   # ⚠️ cambia si usaste timestamp
TEST_PATH  = "data/test_ag.parquet"
RAW_TEST_PATH = "data/test_reviews.csv"      # ⚠️ Ajusta la ruta a tu test_reviews.csv original
OUTPUT_PATH = "submissions/submission_v2.csv"

TARGET_COL = "target"
TEXT_FEATURES = ["categories"]


# ─────────────────────────────────────────────────────────────
# SANITIZE (MISMA QUE TRAIN)
# ─────────────────────────────────────────────────────────────
def sanitize_for_autogluon(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Nullable ints → float32
    nullable_int_cols = [
        c for c in df.columns
        if str(df[c].dtype).startswith("Int")
    ]
    for c in nullable_int_cols:
        df[c] = df[c].astype("float32")

    # TEXT features
    for c in TEXT_FEATURES:
        if c in df.columns:
            df[c] = df[c].astype("string")

    # Object → category
    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    for c in obj_cols:
        if c not in TEXT_FEATURES:
            df[c] = df[c].astype("category")

    return df


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────
def main():
    print("\n🚀 Cargando modelo...")
    predictor = TabularPredictor.load(MODEL_PATH)

    print("📂 Cargando test data...")
    test_data = TabularDataset(TEST_PATH)
    test_data = sanitize_for_autogluon(test_data)

    # Cargar solo los review_id originales para no saturar la RAM
    print("📂 Cargando review_ids originales...")
    original_test_ids = pd.read_csv(RAW_TEST_PATH, usecols=["review_id"])

    # Validar que tengan la misma cantidad de filas
    if len(original_test_ids) != len(test_data):
        print(f"⚠️ ADVERTENCIA: Las filas en {RAW_TEST_PATH} ({len(original_test_ids)}) "
              f"no coinciden con {TEST_PATH} ({len(test_data)})")

    print("🤖 Generando predicciones...")
    preds = predictor.predict(test_data, model=predictor.model_best)

    # Clipping (Yelp: 1–5 estrellas)
    preds = preds.clip(1, 5)

    print("💾 Guardando submission...")
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    # Asignar el review_id real y los valores predichos
    submission = pd.DataFrame({
        "review_id": original_test_ids["review_id"],
        "stars": preds.values # .values asegura que se asigne correctamente ignorando índices de pandas
    })
    
    submission.to_csv(OUTPUT_PATH, index=False)

    print("\n✅ DONE")
    print(f"📄 Archivo: {OUTPUT_PATH}")
    print(f"📊 Stats → min={preds.min():.3f} max={preds.max():.3f} "
          f"mean={preds.mean():.3f} std={preds.std():.3f}")


if __name__ == "__main__":
    main()

Los resultados de la inferencia dan un MAE en test de 0.6519 el cual empeora ligeramente el conseguido con v1 de 0.6393

## V3
### PREPROCESAMIENTO
Mientras que la V2 introdujo la partición temporal para evaluar el modelo correctamente, la V3 lleva el concepto de "tiempo" al interior de las features, convirtiendo datos estáticos en métricas dinámicas que evolucionan junto con el usuario.

- Ingeniería de Características en Ventana Expandible (Expanding Windows): Se reemplazan los promedios globales estáticos por promedios "hasta la fecha".

    - user_avg_stars_at_time / biz_avg_stars_at_time: El modelo ya no ve la nota media total de un usuario o negocio, sino la nota media exacta que tenían en el instante anterior a la reseña actual.

    - user_review_count_at_time / biz_review_count_at_time: Refleja la cantidad de reseñas acumuladas hasta ese momento exacto, indicando la fiabilidad del promedio.

    - delta_stars: Se introduce una métrica relacional que calcula la diferencia directa entre la exigencia histórica del usuario y la calidad histórica del negocio al momento de la interacción.

- Detección de Cold Start (Arranque en Frío): * Creación de flags explícitos (is_cold_user, is_cold_biz) para señalar a la red neuronal cuándo un usuario o negocio no tiene historial previo. Esto permite que el modelo active estrategias diferentes (por ejemplo, basarse exclusivamente en el perfil de precios o categorías del local) cuando las medias históricas son nulas.

- Contextualización del Engagement:

    - user_seniority_days: Se calcula la antigüedad del usuario en el momento exacto de escribir la reseña, reemplazando a la métrica estática y ruidosa de days_yelping.

    - was_elite_at_review: En lugar de usar un conteo total de años como "Elite", se verifica si el usuario ostentaba ese estatus específico en el año en que se publicó la valoración.

- Eliminación Definitiva de Variables Contaminadas: Se descartan proactivamente todas las columnas que contenían información del futuro, agregaciones estáticas globales o métricas post-reseña (average_stars, stars_business, review_count, review_count_business, review_cool, review_funny, review_useful, years_yelping), garantizando un entorno de predicción 100% estricto y libre de fugas de datos (Data Leakage).

In [ ]:
import preprocess_autogluon3
import pandas as pd
# %run preprocess_autogluon3.py
train = pd.read_parquet('data2/train_ag.parquet')
train

### ENTRENAMIENTO

In [ ]:
import train_autogluon3
%run train_autogluon3.py

### INFERENCIA Y RESULTADOS

In [ ]:
import pandas as pd
import os
from autogluon.tabular import TabularPredictor, TabularDataset

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
MODEL_PATH = "models2/autogluon3"   # ⚠️ cambia si usaste timestamp
TEST_PATH  = "data2/test_ag.parquet"
RAW_TEST_PATH = "data2/test_reviews.csv"      # ⚠️ Ajusta la ruta a tu test_reviews.csv original
OUTPUT_PATH = "submissions/submission_v3.csv"

TARGET_COL = "target"
TEXT_FEATURES = ["categories"]


# ─────────────────────────────────────────────────────────────
# SANITIZE (MISMA QUE TRAIN)
# ─────────────────────────────────────────────────────────────
def sanitize_for_autogluon(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Nullable ints → float32
    nullable_int_cols = [
        c for c in df.columns
        if str(df[c].dtype).startswith("Int")
    ]
    for c in nullable_int_cols:
        df[c] = df[c].astype("float32")

    # TEXT features
    for c in TEXT_FEATURES:
        if c in df.columns:
            df[c] = df[c].astype("string")

    # Object → category
    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    for c in obj_cols:
        if c not in TEXT_FEATURES:
            df[c] = df[c].astype("category")

    return df


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────
def main():
    print("\n🚀 Cargando modelo...")
    predictor = TabularPredictor.load(MODEL_PATH)

    print("📂 Cargando test data...")
    test_data = TabularDataset(TEST_PATH)
    test_data = sanitize_for_autogluon(test_data)

    # Cargar solo los review_id originales para no saturar la RAM
    print("📂 Cargando review_ids originales...")
    original_test_ids = pd.read_csv(RAW_TEST_PATH, usecols=["review_id"])

    # Validar que tengan la misma cantidad de filas
    if len(original_test_ids) != len(test_data):
        print(f"⚠️ ADVERTENCIA: Las filas en {RAW_TEST_PATH} ({len(original_test_ids)}) "
              f"no coinciden con {TEST_PATH} ({len(test_data)})")

    print("🤖 Generando predicciones...")
    preds = predictor.predict(test_data, model=predictor.model_best)

    # Clipping (Yelp: 1–5 estrellas)
    preds = preds.clip(1, 5)

    print("💾 Guardando submission...")
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    # Asignar el review_id real y los valores predichos
    submission = pd.DataFrame({
        "review_id": original_test_ids["review_id"],
        "stars": preds.values # .values asegura que se asigne correctamente ignorando índices de pandas
    })
    
    submission.to_csv(OUTPUT_PATH, index=False)

    print("\n✅ DONE")
    print(f"📄 Archivo: {OUTPUT_PATH}")
    print(f"📊 Stats → min={preds.min():.3f} max={preds.max():.3f} "
          f"mean={preds.mean():.3f} std={preds.std():.3f}")


if __name__ == "__main__":
    main()

Los resultados de la inferencia dan un MAE de test de 1.3866 el cual es el peor hasta ahora.

### COMPROBACIÓN DE CORRELACIÓN DE MALOS RESULTADOS CON EL TEMPORAL SPLIT
Entrenaremos con un split aleatorio sin tener en cuenta el tiempo para comprobar si la razón de los malos rendimientos es el temporal split o los campos eliminados/añadidos en las versiones v2 y v3.